# Compression, Materialized Views, and Inverted Indexes

**Module 6: Table Properties, Materialized Views, and Indexes** · [Quiz](quiz06_properties_mv_and_indexes.ipynb) · [Course README](../../README.md)

This Lab works on the telemetry that the earlier modules loaded into Doris: the logs in `otel_logs` and the gauge metrics in `otel_metrics_gauge`.

### Initialize the Lab

Run this cell once before using `lab.shell(...)` or `lab.sql(...)`.


In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)


## Check the logs in Doris

The examples below read `otel_logs`: the demo's logs from Module 1 and the FE log from Module 2. Count the rows by service:

In [ ]:
log_services = lab.wait_for_rows(
    """
    SELECT service_name, COUNT(*) AS log_rows, MAX(timestamp) AS latest_log
    FROM otel_logs
    GROUP BY service_name
    ORDER BY log_rows DESC
    """,
    title="Log rows by service in otel_logs",
    hint="otel_logs has no rows. Start the DOG stack and the demo with the cells in Module 1, "
         "and load the FE log with section 1 of Module 2. Then rerun this cell.",
)

**Expected result**

One row per service: `doris-fe`, the FE log that Module 2 loaded, and the demo's services, such as `frontend-proxy`, `kafka`, `currency`, and `cart`. `load-generator`, which the inverted-index example at the end filters on, has a few rows from when the demo started. If `doris-fe` is the only service, the demo has not sent logs: start it with `lab.start_demo()` in Module 1 and rerun this cell after a minute.

## Compress

Observability systems may receive a very large amount of data every day. In addition to data governance, what other methods can help reduce disk usage? One recommended approach is to use the ZSTD compression algorithm to compress data efficiently and reduce storage consumption.

The logs have already been imported into the `otel_logs` table. That table uses the default compression algorithm, LZ4F. To compare it with the default `otel_logs` table, create a new table named `otel_logs_zstd`:

In [ ]:
lab.execute(r"""
CREATE TABLE `otel_logs_zstd` (
  `timestamp` datetime(6) NULL,
  `service_name` varchar(200) NULL,
  `service_instance_id` varchar(200) NULL,
  `trace_id` varchar(200) NULL,
  `span_id` text NULL,
  `severity_number` int NULL,
  `severity_text` text NULL,
  `body` text NULL,
  `resource_attributes` variant NULL,
  `log_attributes` variant NULL,
  `scope_name` text NULL,
  `scope_version` text NULL,
  INDEX idx_service_name (`service_name`) USING INVERTED,
  INDEX idx_timestamp (`timestamp`) USING INVERTED,
  INDEX idx_service_instance_id (`service_instance_id`) USING INVERTED,
  INDEX idx_trace_id (`trace_id`) USING INVERTED,
  INDEX idx_span_id (`span_id`) USING INVERTED,
  INDEX idx_severity_number (`severity_number`) USING INVERTED,
  INDEX idx_body (`body`) USING INVERTED PROPERTIES("lower_case" = "true", "parser" = "unicode", "support_phrase" = "true"),
  INDEX idx_severity_text (`severity_text`) USING INVERTED,
  INDEX idx_resource_attributes (`resource_attributes`) USING INVERTED,
  INDEX idx_log_attributes (`log_attributes`) USING INVERTED,
  INDEX idx_scope_name (`scope_name`) USING INVERTED,
  INDEX idx_scope_version (`scope_version`) USING INVERTED
) ENGINE=OLAP
DUPLICATE KEY(`timestamp`, `service_name`)
PARTITION BY RANGE(`timestamp`)
(PARTITION p20260908 VALUES [('2026-09-08 00:00:00'), ('2026-09-09 00:00:00')),
PARTITION p20260909 VALUES [('2026-09-09 00:00:00'), ('2026-09-10 00:00:00')),
PARTITION p20260910 VALUES [('2026-09-10 00:00:00'), ('2026-09-11 00:00:00')),
PARTITION p20260911 VALUES [('2026-09-11 00:00:00'), ('2026-09-12 00:00:00')),
PARTITION p20260912 VALUES [('2026-09-12 00:00:00'), ('2026-09-13 00:00:00')),
PARTITION p20260913 VALUES [('2026-09-13 00:00:00'), ('2026-09-14 00:00:00')),
PARTITION p20260914 VALUES [('2026-09-14 00:00:00'), ('2026-09-15 00:00:00')),
PARTITION p20260915 VALUES [('2026-09-15 00:00:00'), ('2026-09-16 00:00:00')),
PARTITION p20260916 VALUES [('2026-09-16 00:00:00'), ('2026-09-17 00:00:00')),
PARTITION p20260917 VALUES [('2026-09-17 00:00:00'), ('2026-09-18 00:00:00')),
PARTITION p20260918 VALUES [('2026-09-18 00:00:00'), ('2026-09-19 00:00:00')),
PARTITION p20260919 VALUES [('2026-09-19 00:00:00'), ('2026-09-20 00:00:00')))
DISTRIBUTED BY RANDOM BUCKETS AUTO
PROPERTIES (
"replication_allocation" = "tag.location.default: 1",
"min_load_replica_num" = "-1",
"is_being_synced" = "false",
"dynamic_partition.enable" = "true",
"dynamic_partition.time_unit" = "DAY",
"dynamic_partition.time_zone" = "Etc/UTC",
"dynamic_partition.start" = "-10",
"dynamic_partition.end" = "1",
"dynamic_partition.prefix" = "p",
"dynamic_partition.replication_allocation" = "tag.location.default: 1",
"dynamic_partition.buckets" = "10",
"dynamic_partition.create_history_partition" = "true",
"dynamic_partition.history_partition_num" = "10",
"dynamic_partition.hot_partition_num" = "0",
"dynamic_partition.reserved_history_periods" = "NULL",
"dynamic_partition.storage_policy" = "",
"storage_medium" = "hdd",
"storage_format" = "V2",
"inverted_index_storage_format" = "V2",
"light_schema_change" = "true",
"compaction_policy" = "time_series",
"time_series_compaction_goal_size_mbytes" = "1024",
"time_series_compaction_file_count_threshold" = "2000",
"time_series_compaction_time_threshold_seconds" = "3600",
"time_series_compaction_empty_rowsets_threshold" = "5",
"time_series_compaction_level_threshold" = "1",
"disable_auto_compaction" = "false",
"group_commit_interval_ms" = "10000",
"group_commit_data_bytes" = "134217728",
"compression" = "zstd"
);
""", title='ZSTD-compressed log table')


Note the final property, `"compression" = "zstd"`. It specifies that the ZSTD compression algorithm is used.

Next, insert the data from `otel_logs` into `otel_logs_zstd`:

In [ ]:
lab.execute("INSERT INTO otel_logs_zstd SELECT * FROM otel_logs")


Because the data files are not large, this example is intended only to show the difference. In scenarios with larger data volumes, ZSTD can provide better compression results:

<img src="images/compress.png" alt="Comparison of LZ4F and ZSTD compression in Doris" style="max-width:100%;height:auto;">

## Materialized Views

In metrics scenarios, if you need to analyze the maximum value of certain `metric_name` values over a historical period, you can query the `otel_metrics_gauge` table directly. However, this table stores the original metrics data, and direct aggregation can be inefficient. Doris materialized views can be used to accelerate these queries.

The `otel_metrics_gauge` table has already been created. You only need to create a materialized view on this table to accelerate the required query:

In [ ]:
lab.execute(r"""
CREATE MATERIALIZED VIEW mv_for_metrics AS
SELECT metric_name AS mn, MAX(value) AS max_value
FROM otel_metrics_gauge
GROUP BY metric_name;
""", title='Materialized view for metric maxima')


After the materialized view is created, you can query the original table without needing to remember that the materialized view exists:

In [ ]:
lab.sql(r"""
SELECT metric_name, MAX(value) FROM otel_metrics_gauge GROUP BY metric_name LIMIT 5;
""", title='Metric maximum query')


This statement will hit the materialized view. Because the materialized view already contains aggregated data, the query will be accelerated. You can use `EXPLAIN` to confirm this:

<img src="images/explain.png" alt="EXPLAIN output showing that the materialized view is used" style="max-width:100%;height:auto;">

## Inverted Indexes

You may also have noticed that many of the preceding table definitions contain statements such as the following. These statements create inverted indexes:

The original log table contains the following inverted indexes:

```sql
...
  INDEX idx_service_name (`service_name`) USING INVERTED,
  INDEX idx_timestamp (`timestamp`) USING INVERTED,
  INDEX idx_service_instance_id (`service_instance_id`) USING INVERTED,
  INDEX idx_trace_id (`trace_id`) USING INVERTED,
  ...
```


How can you determine whether these indexes are being used and whether they reduce the amount of data that must be scanned?

Consider a simple scenario where you need to retrieve detailed logs whose `service_name` equals a specific value. The following statement can be generated quickly. The first two session variables enable profiling and set the profile level:

In [ ]:
lab.execute("SET enable_profile = true")
lab.sql(r"""
SELECT * FROM otel_logs WHERE service_name = 'load-generator' LIMIT 10
""", title="Query with profile collection enabled")


Use the following statement to find the profile ID for the preceding query:

In [ ]:
lab.sql(r"""
SHOW QUERY PROFILE
""", title='Recent query profiles')


On the FE that executed the query, run the following command:

In [ ]:
PROFILE_ID = "88e5c13d303b4385-8b6814758777d498	"
assert not PROFILE_ID.startswith("<"), "Replace PROFILE_ID with the ID shown by SHOW QUERY PROFILE."

lab.shell(f"""set -euo pipefail
curl -uroot: "{lab.fe_http_url}/api/profile/text?query_id=88e5c13d303b4385-8b6814758777d498"
""", title="Fetch the Doris query profile")


Search for the keyword `RowsInvertedIndexFiltered`. Information similar to the following is displayed:

```text
RowsInvertedIndexFiltered: 11.702K (11702)
```

This indicates that 11,702 rows were filtered out by the inverted index. The index filtered part of the data and improved query efficiency.

## Next

Open [Quiz 6: Table Properties, Materialized Views, and Indexes](quiz06_properties_mv_and_indexes.ipynb), then continue to [Module 7: Trace Analysis and Litefuse](../module07-data-analysis-litefuse/lab07_data_analysis_litefuse.ipynb).